# SAE Dual-Pipeline Visualization

这个 notebook 读取 `scripts/run_sae_pipeline.py` 生成的 `pipeline_summary.json`，对 `standard` 和 `attnres` 两个模型的 SAE 训练曲线与结构评估结果做并排可视化。

## 1. 定位结果目录

这一步不是可视化本身，而是先自动定位仓库根目录和 `pipeline_summary.json`。

你可以把这里理解成“告诉 notebook 去哪里找刚刚跑完的 SAE 双模型结果”。如果路径不对，后面的图都不会画出来。

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

plt.rcParams["axes.unicode_minus"] = False
MODEL_COLORS = {
    "standard": "#1f77b4",
    "attnres": "#ff7f0e",
}


def find_repo_root(start: Path | None = None) -> Path:
    candidate = Path.cwd() if start is None else start
    for path in [candidate, *candidate.parents]:
        if (path / "stream_analysis").exists() and (path / "scripts").exists():
            return path
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
SUMMARY_PATH = REPO_ROOT / "outputs" / "sae_dual_pipeline" / "pipeline_summary.json"
REPO_ROOT, SUMMARY_PATH

## 2. 加载总结果清单

这一步会读取 `pipeline_summary.json`，然后把两个模型最关键的输出路径和最优验证集重构误差整理成一个表。

这张表的作用是先做一次“结果核对”：确认 `standard` 和 `attnres` 的 checkpoint、SAE 输出目录、评估 summary 都已经生成。

In [ ]:
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Missing pipeline summary: {SUMMARY_PATH}\n"
        "先运行 accelerate launch scripts/run_sae_pipeline.py ... 生成双模型 SAE 结果。"
    )

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    pipeline_summary = json.load(handle)

overview_rows = []
for model_type, payload in pipeline_summary["models"].items():
    overview_rows.append(
        {
            "model_type": model_type,
            "checkpoint": payload["checkpoint_path"],
            "best_val_recon_mse": payload["best_val_recon_mse"],
            "sae_dir": payload["sae_dir"],
            "eval_summary_path": payload.get("eval_summary_path", ""),
        }
    )

if pd is not None:
    display(pd.DataFrame(overview_rows))
else:
    overview_rows

## 3. 读取训练曲线和评估 summary

这一步还是准备数据，不会直接画图。它会把每个模型的 `metrics.csv` 和 `eval_summary.json` 读进内存。

后面的所有可视化都依赖这一步输出的 `train_curves` 和 `eval_payloads`。

In [ ]:
def read_metrics_csv(path: str | Path):
    rows = []
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            rows.append({key: float(value) for key, value in row.items()})
    return rows


DECODER_DISPLAY_MAX_LATENTS = None
DECODER_OVERLAP_QUANTILES = (0.01, 0.99)
DECODER_OVERLAP_TOPK_HOTSPOTS = 75
DECODER_SUBMATRIX_TOPK_PAIRS = 100
DECODER_SUBMATRIX_ORDERING_METHOD = "mean_overlap"
DECODER_SUBMATRIX_MAX_TICKS = 20


def load_eval_summary(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_decoder_overlap_matrix(checkpoint_path: str | Path):
    payload = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=False)
    w_dec = payload["model_state"]["W_dec"].detach().to(dtype=torch.float32)
    w_dec = w_dec / w_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
    return (w_dec @ w_dec.T).cpu().numpy()


def maybe_subsample_matrix_for_display(matrix: np.ndarray, max_latents: int | None = DECODER_DISPLAY_MAX_LATENTS):
    if max_latents is None or matrix.shape[0] <= max_latents:
        return matrix, None
    indices = np.linspace(0, matrix.shape[0] - 1, num=max_latents, dtype=int)
    return matrix[np.ix_(indices, indices)], indices


def mask_diagonal_for_display(matrix: np.ndarray) -> np.ndarray:
    masked = np.array(matrix, dtype=float, copy=True)
    np.fill_diagonal(masked, np.nan)
    return masked


def offdiag_values(matrix: np.ndarray) -> np.ndarray:
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f"Expected a square matrix, got {matrix.shape}.")
    mask = ~np.eye(matrix.shape[0], dtype=bool)
    values = np.asarray(matrix[mask], dtype=float)
    return values[np.isfinite(values)]


def compute_robust_color_limits(
    matrix: np.ndarray,
    lower_q: float = DECODER_OVERLAP_QUANTILES[0],
    upper_q: float = DECODER_OVERLAP_QUANTILES[1],
):
    values = offdiag_values(matrix)
    if values.size == 0:
        raise RuntimeError("No finite off-diagonal values found for decoder overlap heatmap.")
    vmin = float(np.quantile(values, lower_q))
    vmax = float(np.quantile(values, upper_q))
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        vmin = float(np.nanmin(values))
        vmax = float(np.nanmax(values))
    if vmin >= vmax:
        center = float(np.nanmedian(values))
        spread = max(float(np.nanstd(values)), 1e-6)
        vmin = center - spread
        vmax = center + spread
    return vmin, vmax


def get_topk_offdiag_pairs(matrix: np.ndarray, topk: int = DECODER_OVERLAP_TOPK_HOTSPOTS):
    if topk <= 0:
        return np.empty((0,), dtype=int), np.empty((0,), dtype=int), np.empty((0,), dtype=float)
    rows, cols = np.triu_indices(matrix.shape[0], k=1)
    values = np.asarray(matrix[rows, cols], dtype=float)
    finite_mask = np.isfinite(values)
    rows = rows[finite_mask]
    cols = cols[finite_mask]
    values = values[finite_mask]
    if values.size == 0:
        return np.empty((0,), dtype=int), np.empty((0,), dtype=int), np.empty((0,), dtype=float)
    topk = min(topk, values.size)
    keep = np.argpartition(values, -topk)[-topk:]
    order = keep[np.argsort(values[keep])[::-1]]
    return rows[order], cols[order], values[order]


def collect_latents_from_pairs(rows: np.ndarray, cols: np.ndarray) -> np.ndarray:
    if rows.size == 0 or cols.size == 0:
        return np.empty((0,), dtype=int)
    return np.unique(np.concatenate([rows, cols]))


def order_selected_latents(
    submatrix: np.ndarray,
    latent_ids: np.ndarray,
    method: str = DECODER_SUBMATRIX_ORDERING_METHOD,
) -> np.ndarray:
    if latent_ids.size == 0:
        return latent_ids
    if method != "mean_overlap":
        raise ValueError(f"Unsupported ordering method: {method}")
    masked = mask_diagonal_for_display(submatrix)
    scores = np.nanmean(masked, axis=1)
    scores = np.where(np.isfinite(scores), scores, -np.inf)
    order = np.argsort(scores)[::-1]
    return latent_ids[order]


def build_top_pair_induced_submatrix(
    matrix: np.ndarray,
    top_k: int = DECODER_SUBMATRIX_TOPK_PAIRS,
    ordering_method: str = DECODER_SUBMATRIX_ORDERING_METHOD,
):
    pair_rows, pair_cols, pair_values = get_topk_offdiag_pairs(matrix, topk=top_k)
    selected_latents = collect_latents_from_pairs(pair_rows, pair_cols)
    if selected_latents.size == 0:
        raise RuntimeError("No off-diagonal decoder-overlap pairs were selected.")
    unsorted_submatrix = np.asarray(matrix[np.ix_(selected_latents, selected_latents)], dtype=float)
    ordered_latents = order_selected_latents(
        unsorted_submatrix,
        selected_latents,
        method=ordering_method,
    )
    ordered_submatrix = np.asarray(matrix[np.ix_(ordered_latents, ordered_latents)], dtype=float)
    return {
        "pair_rows": pair_rows,
        "pair_cols": pair_cols,
        "pair_values": pair_values,
        "selected_latents": selected_latents,
        "ordered_latents": ordered_latents,
        "ordered_submatrix": ordered_submatrix,
        "ordering_method": ordering_method,
        "top_k": top_k,
    }


def configure_sparse_ticks(axis, size: int, sampled_indices: np.ndarray | None = None, tick_count: int = 6):
    if size <= 1:
        axis.set_xticks([0])
        axis.set_yticks([0])
        return
    ticks = np.linspace(0, size - 1, num=min(tick_count, size), dtype=int)
    axis.set_xticks(ticks)
    axis.set_yticks(ticks)
    labels = sampled_indices[ticks] if sampled_indices is not None else ticks
    axis.set_xticklabels(labels)
    axis.set_yticklabels(labels)


def plot_decoder_overlap_heatmap(
    matrix: np.ndarray,
    *,
    model_type: str,
    sampled_indices: np.ndarray | None = None,
    lower_q: float = DECODER_OVERLAP_QUANTILES[0],
    upper_q: float = DECODER_OVERLAP_QUANTILES[1],
    topk_hotspots: int = DECODER_OVERLAP_TOPK_HOTSPOTS,
):
    display_matrix = np.asarray(matrix, dtype=float)
    masked_matrix = mask_diagonal_for_display(display_matrix)
    vmin, vmax = compute_robust_color_limits(display_matrix, lower_q=lower_q, upper_q=upper_q)
    rows, cols, hotspot_values = get_topk_offdiag_pairs(display_matrix, topk=topk_hotspots)
    offdiag = offdiag_values(display_matrix)

    print(f"[{model_type}] decoder overlap off-diagonal summary")
    print(f"  display_shape: {display_matrix.shape}")
    print(f"  offdiag_mean: {float(np.mean(offdiag)):.6f}")
    print(f"  offdiag_median: {float(np.median(offdiag)):.6f}")
    print(f"  offdiag_max: {float(np.max(offdiag)):.6f}")
    print(f"  robust_vmin: {vmin:.6f}")
    print(f"  robust_vmax: {vmax:.6f}")
    if hotspot_values.size > 0:
        print(f"  topk_hotspots: {hotspot_values.size}")
        print(f"  topk_value_range: [{float(np.min(hotspot_values)):.6f}, {float(np.max(hotspot_values)):.6f}]")
    else:
        print("  topk_hotspots: 0")

    cmap = plt.cm.magma.copy()
    cmap.set_bad(color="#f2f2f2")
    fig, axis = plt.subplots(figsize=(8.8, 7.6), dpi=220)
    image = axis.imshow(
        masked_matrix,
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        origin="upper",
    )
    if hotspot_values.size > 0:
        overlay_rows = np.concatenate([rows, cols])
        overlay_cols = np.concatenate([cols, rows])
        axis.scatter(
            overlay_cols,
            overlay_rows,
            s=6,
            facecolors="none",
            edgecolors="#7fffd4",
            linewidths=0.25,
            alpha=0.85,
        )
    title = f"{model_type}: decoder overlap heatmap (off-diagonal, robust scale)"
    if sampled_indices is not None:
        title += f" [display sampled to {display_matrix.shape[0]} latents]"
    axis.set_title(title)
    axis.set_xlabel("latent index j")
    axis.set_ylabel("latent index i")
    configure_sparse_ticks(axis, display_matrix.shape[0], sampled_indices=sampled_indices, tick_count=6)
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label("cosine overlap (off-diagonal, robust scale)")
    plt.tight_layout()
    plt.show()


def plot_latent_submatrix_heatmap(
    matrix: np.ndarray,
    *,
    model_type: str,
    top_k: int = DECODER_SUBMATRIX_TOPK_PAIRS,
    ordering_method: str = DECODER_SUBMATRIX_ORDERING_METHOD,
    lower_q: float = DECODER_OVERLAP_QUANTILES[0],
    upper_q: float = DECODER_OVERLAP_QUANTILES[1],
    max_ticks: int = DECODER_SUBMATRIX_MAX_TICKS,
):
    bundle = build_top_pair_induced_submatrix(
        matrix,
        top_k=top_k,
        ordering_method=ordering_method,
    )
    ordered_latents = bundle["ordered_latents"]
    submatrix = bundle["ordered_submatrix"]
    masked_submatrix = mask_diagonal_for_display(submatrix)
    vmin, vmax = compute_robust_color_limits(submatrix, lower_q=lower_q, upper_q=upper_q)
    offdiag = offdiag_values(submatrix)

    print(f"[{model_type}] top-pair-induced submatrix summary")
    print(f"  top_k: {bundle['top_k']}")
    print(f"  selected_latent_count: {ordered_latents.size}")
    print(f"  submatrix_shape: {submatrix.shape}")
    print(f"  ordering_method: {bundle['ordering_method']}")
    print(f"  submatrix_offdiag_mean: {float(np.mean(offdiag)):.6f}")
    print(f"  submatrix_offdiag_max: {float(np.max(offdiag)):.6f}")
    print(f"  robust_vmin: {vmin:.6f}")
    print(f"  robust_vmax: {vmax:.6f}")
    preview_count = min(10, bundle['pair_values'].size)
    print(f"  involved_latent_count: {bundle['selected_latents'].size}")
    for idx in range(preview_count):
        print(
            f"  top_pair_{idx + 1}: ({int(bundle['pair_rows'][idx])}, {int(bundle['pair_cols'][idx])}) -> {float(bundle['pair_values'][idx]):.6f}"
        )

    cmap = plt.cm.magma.copy()
    cmap.set_bad(color="#f2f2f2")
    fig, axis = plt.subplots(figsize=(8.4, 7.4), dpi=220)
    image = axis.imshow(
        masked_submatrix,
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        origin="upper",
    )
    axis.set_title(
        f"{model_type}: top-pair-induced overlap submatrix (top_k={top_k}, order={ordering_method})"
    )
    axis.set_xlabel("ordered selected latent index j")
    axis.set_ylabel("ordered selected latent index i")
    configure_sparse_ticks(
        axis,
        submatrix.shape[0],
        sampled_indices=ordered_latents,
        tick_count=max_ticks,
    )
    for tick in axis.get_xticklabels():
        tick.set_rotation(90)
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label("cosine overlap (submatrix off-diagonal, robust scale)")
    plt.tight_layout()
    plt.show()


train_curves = {
    model_type: read_metrics_csv(payload["metrics_path"])
    for model_type, payload in pipeline_summary["models"].items()
}
eval_payloads = {
    model_type: load_eval_summary(payload["eval_summary_path"])
    for model_type, payload in pipeline_summary["models"].items()
}

list(train_curves.keys())

## 4. 训练曲线可视化

这张图展示两个模型在 SAE 训练过程中的重构误差变化：

- 左图是 `train_recon_mse`，看训练集上是否稳定下降
- 右图是 `val_recon_mse`，看 held-out 验证激活上的泛化情况

如果右图持续下降且没有明显发散，通常说明这组 SAE 超参数是可用的；如果训练误差很低但验证误差不降，说明可能过拟合或训练数据分布不够稳。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
for axis, metric_key, title in [
    (axes[0], "train_recon_mse", "Train Reconstruction MSE"),
    (axes[1], "val_recon_mse", "Validation Reconstruction MSE"),
]:
    for model_type, rows in train_curves.items():
        steps = [int(row["step"]) for row in rows]
        values = [row[metric_key] for row in rows]
        axis.plot(steps, values, marker="o", label=model_type, color=MODEL_COLORS.get(model_type, "#4c4c4c"))
    axis.set_title(title)
    axis.set_xlabel("step")
    axis.set_ylabel(metric_key)
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

## 5. SAE 结构评估指标对比

这一部分会把两个模型最终 SAE 的核心结构指标拆成多张独立柱状图分别比较：

- `recon_mse`：原始重构误差，越低越好
- `normalized_recon_mse`：按输入方差归一化后的重构误差，越低越好
- `avg_l0`：平均激活 latent 数，反映稀疏程度
- `dead_latent_frac`：死特征比例，越低通常越好

这样做的好处是不会把量纲不同的指标挤在同一张图里，更容易单独判断某一个指标上到底是哪边更好。

In [ ]:
metric_specs = [
    ("recon_mse", "Reconstruction MSE", "Lower is better; raw reconstruction error."),
    ("normalized_recon_mse", "Normalized Reconstruction MSE", "Lower is better; normalized by input variance."),
    ("avg_l0", "Average L0", "Average number of active latents per sample."),
    ("dead_latent_frac", "Dead Latent Fraction", "Lower is better; fraction of inactive latents."),
]
model_order = list(pipeline_summary["models"].keys())

for metric_key, title, subtitle in metric_specs:
    values = [float(eval_payloads[model_type]["summary"][metric_key]) for model_type in model_order]
    x = np.arange(len(model_order))
    colors = [MODEL_COLORS.get(model_type, "#4c4c4c") for model_type in model_order]

    fig, ax = plt.subplots(figsize=(7, 4.8))
    bars = ax.bar(x, values, width=0.6, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(model_order)
    ax.set_ylabel(metric_key)
    ax.set_title(f"{title}\n{subtitle}")
    ax.grid(axis="y", alpha=0.3)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

## 6. Decoder Overlap 热图

这张热图展示 SAE decoder 不同 latent 原子之间的余弦相似度。

你可以把它理解成“不同特征之间是不是在重复表示类似方向”。

- 对角线附近或大面积高亮：说明很多 latent 很相似，特征有冗余
- 整体更分散、非对角线不那么亮：说明 decoder 原子更分化

这张图主要是在看特征字典是否“分得开”。

这个优化不改变 overlap 结果本身，只改善显示动态范围和可读性。

主对角线的 self-overlap=1 和大量接近 0 的背景值，会淹没真正重要的 off-diagonal 结构；因此这里会在可视化阶段 mask 主对角线，并用 off-diagonal 分位数做 robust scaling。

图上还会可选叠加 top-k off-diagonal hotspot，帮助你直接看到 overlap 最高的 latent pair。

如果 latent 数特别大，想让图更轻一点，可以把前面代码里的 `DECODER_DISPLAY_MAX_LATENTS` 改成一个整数，比如 `768` 或 `1024`；这只影响显示，不改原矩阵。

In [ ]:
decoder_overlap_matrices = {}

for model_type in model_order:
    checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]
    overlap_matrix = load_decoder_overlap_matrix(checkpoint_path)
    decoder_overlap_matrices[model_type] = overlap_matrix
    display_matrix, sampled_indices = maybe_subsample_matrix_for_display(overlap_matrix)
    plot_decoder_overlap_heatmap(
        display_matrix,
        model_type=model_type,
        sampled_indices=sampled_indices,
    )

## 6.1 Top-Pair-Induced Latent Submatrix Heatmap

这张图不是再画整个全量 overlap 矩阵，而是：

1. 先从 off-diagonal overlap 里找 top-k highest-overlap pairs
2. 收集这些 pair 涉及到的 latent
3. 用这些 latent 构成更小的 overlap 子矩阵
4. 再按 `mean_overlap` 排序后画热力图

这个图更适合回答：高 overlap latent 是不是集中在某个局部小群里，以及这些小群之间有没有可见的 block 结构。

这里同样只改显示方式，不改 overlap 数值本身。

In [ ]:
for model_type in model_order:
    overlap_matrix = decoder_overlap_matrices.get(model_type)
    if overlap_matrix is None:
        checkpoint_path = pipeline_summary["models"][model_type]["sae_best_checkpoint"]
        overlap_matrix = load_decoder_overlap_matrix(checkpoint_path)
        decoder_overlap_matrices[model_type] = overlap_matrix
    plot_latent_submatrix_heatmap(
        overlap_matrix,
        model_type=model_type,
        top_k=DECODER_SUBMATRIX_TOPK_PAIRS,
        ordering_method=DECODER_SUBMATRIX_ORDERING_METHOD,
        max_ticks=DECODER_SUBMATRIX_MAX_TICKS,
    )

## 7. Coactivation 热图

这张热图展示 latent 两两之间在样本上共同激活的频率。

它回答的是：哪些特征经常一起出现，是否形成了稳定的组合结构。

- 某些局部块特别亮：说明有一组 latent 经常成团共现
- 整体比较稀疏：说明特征之间更独立

这张图更偏向看“特征之间的协同关系”，而不是单个特征本身质量。

In [ ]:
fig, axes = plt.subplots(1, len(model_order), figsize=(7 * len(model_order), 5))
if len(model_order) == 1:
    axes = [axes]

for axis, model_type in zip(axes, model_order):
    coactivation = eval_payloads[model_type]["coactivation"].get("heatmap")
    if coactivation is None:
        axis.axis("off")
        axis.set_title(f"{model_type}: coactivation heatmap unavailable")
        continue
    image = axis.imshow(np.asarray(coactivation, dtype=float), aspect="auto", interpolation="nearest")
    axis.set_title(f"{model_type}: coactivation")
    axis.set_xlabel("latent j")
    axis.set_ylabel("latent i")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()